In [ ]:
!pip install -r requirements.txt
!pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121
!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.7/flash_attn-2.5.7+cu122torch2.1cxx11abiFALSE-cp39-cp39-linux_x86_64.whl
!pip install transformers==4.44.2
!pip install triton==2.2.0
!pip install accelerate
!pip install flash_attn
!pip install datasets

In [1]:
!jupyter nbextension enable --py widgetsnbextension

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


# 下载数据集

In [ ]:
!wget -O video.zip https://huggingface.co/datasets/omni-research/DREAM-1K/resolve/main/video/video.zip
!apt-get update && apt-get install unzip
!unzip video.zip -d ./data/video

# 模型推理

In [ ]:
from tasks.utils import load_model_and_processor

model, processor = load_model_and_processor("omni-research/Tarsier-7b", 8)

generate_kwargs = {
    "do_sample": False,
    "max_new_tokens": 512,
    "top_p": 1,
    "temperature": 0,
    "use_cache": True
}

input_file = r"/root/tarsier/assets/videos/demo_cli_example.mp4"
prompt = "<video>\nDescribe the video in detail."

inputs = processor(prompt, input_file, edit_prompt=True, return_prompt=True)
if 'prompt' in inputs:
    print(f"Prompt: {inputs.pop('prompt')}")
inputs = {k:v.to(model.device) for k,v in inputs.items() if v is not None}
outputs = model.generate(
    **inputs,
    **generate_kwargs,
)
output_text = processor.tokenizer.decode(outputs[0][inputs['input_ids'][0].shape[0]:], skip_special_tokens=True)

In [3]:
from datasets import load_dataset
dataset = load_dataset('json', data_files=r"/root/tarsier/data/annotations/DREAM-1k-copy.jsonl")

In [ ]:
dataset['train']

In [ ]:
dataset['train']['video_file'][:10]

In [ ]:
dataset['train']['text'][:10]

In [1]:
from tasks.utils import load_model_and_processor

model, processor = load_model_and_processor("omni-research/Tarsier-7b", 8)

Load model and processor from: omni-research/Tarsier-7b; with max_n_frames=8
### do_image_padding is set as False, images will be resized directly!


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
You are attempting to use Flash Attention 2.0 without specifying a torch dtype. This might lead to unexpected behaviour
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [71]:
dataset['train']['text'][1]

{'prompt': '<video>\nDescribe the video in detail.',
 'response': 'One character raises their hand and then puts it down. The other character walks away and is followed by the former, both characters go up a staircase together.'}

In [7]:
IGNORE_INDEX = -100

In [9]:
import torch
s = processor(dataset['train']['text'][0]['prompt'], dataset['train']['video_file'][0], edit_prompt=True)
t = dataset['train']['text'][0]['response']
inputs = {k:v for k,v in s.items() if v is not None}
source_id = inputs['input_ids'][0]
target_id = processor.tokenizer.encode(t + '</s>', add_special_tokens=False, return_tensors='pt')[0]
input_id = torch.concat((source_id, target_id), dim=0)
label = input_id.clone()
label[:len(source_id)] = IGNORE_INDEX

In [10]:
source_id

tensor([    1,  3148,  1001, 29901, 29871, 32000, 32000, 32000, 32000, 32000,
        32000, 32000, 32000, 29871,    13,  4002, 29581,   278,  4863,   297,
         9493, 29889,   319,  1799,  9047, 13566, 29901, 29871])

In [11]:
input_id

tensor([    1,  3148,  1001, 29901, 29871, 32000, 32000, 32000, 32000, 32000,
        32000, 32000, 32000, 29871,    13,  4002, 29581,   278,  4863,   297,
         9493, 29889,   319,  1799,  9047, 13566, 29901, 29871,   319,  2931,
        11176,  2710,   515, 25158,   322,  3974, 29892,   769,  4203,   889,
         2152,   373,   278,  5962,   322,  1754,   263, 29502,   304,  2367,
        11994, 29892,  1550,  1790,  2318,   310,  4890,   364, 15392,  6375,
          322,  2646,  1327,   287,  1023,  7933,  1601, 23080, 29889,     2])

In [12]:
label

tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,   319,  2931,
        11176,  2710,   515, 25158,   322,  3974, 29892,   769,  4203,   889,
         2152,   373,   278,  5962,   322,  1754,   263, 29502,   304,  2367,
        11994, 29892,  1550,  1790,  2318,   310,  4890,   364, 15392,  6375,
          322,  2646,  1327,   287,  1023,  7933,  1601, 23080, 29889,     2])

In [87]:
label

tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  2931, 11176,
         2710,   515, 25158,   322,  3974, 29892,   769,  4203,   889,  2152,
          373,   278,  5962,   322,  1754,   263, 29502,   304,  2367, 11994,
        29892,  1550,  1790,  2318,   310,  4890,   364, 15392,  6375,   322,
         2646,  1327,   287,  1023,  7933,  1601, 23080, 29889,     2])